# Face Attribute Detection with UniFace

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yakhyo/uniface/blob/main/examples/14_face_attributes.ipynb)

FaceAttribNet reads five independent binary states per face: left and right eye openness, eyeglasses, face mask, and sunglasses.

<img src="https://raw.githubusercontent.com/yakhyo/uniface/main/assets/demo/face_states_alt.jpg" width="100%">

---

**UniFace** is a lightweight, production-ready Python library for face detection, recognition, tracking, landmark analysis, face parsing, gaze estimation, and face attributes.

GitHub: [github.com/yakhyo/uniface](https://github.com/yakhyo/uniface) | Docs: [yakhyo.github.io/uniface](https://yakhyo.github.io/uniface)

## Setup


In [ ]:
%pip install -q "uniface[cpu]"

# Clone repo for assets (Colab only)
import os
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    if not os.path.exists('uniface'):
        !git clone --depth 1 https://github.com/yakhyo/uniface.git
    os.chdir('uniface/examples')

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

import uniface
from uniface.attribute import FaceAttribNet
from uniface.detection import RetinaFace

print(f"UniFace version: {uniface.__version__}")

In [ ]:
# Initialize face detector
detector = RetinaFace(confidence_threshold=0.5)

# Initialize face attribute model
face_attrib = FaceAttribNet()

print("Models initialized successfully!")

## 1. Process All Test Images

Each face gets a `FaceStateResult` with five **independent** probabilities: `left_eye_open`, `right_eye_open`, `eyeglasses`, `mask`, `sunglasses`.

> These come from independent binary heads. They do not sum to 1 and several can be high at once (a face can wear both sunglasses and a mask). Threshold each attribute separately; never `argmax`.

In [ ]:
THRESHOLD = 0.5
CROP = 512

# the source photos differ in aspect ratio and framing, so crop each to the same square
SUBJECTS = [
    ('age_adult.jpg', 'eyes open'),
    ('mesh_face.jpg', 'no accessories'),
    ('state_b_glasses.jpg', 'glasses'),
    ('state_b_sunglasses.jpg', 'sunglasses'),
    ('state_b_mask.jpg', 'mask'),
]
demo_dir = Path('../assets/source')


def face_panel(path, margin=0.5):
    """Square crop centred on the largest face, so every panel is the same size."""
    image = cv2.imread(str(path))
    faces = detector.detect(image)
    if not faces:
        return None, None
    face = max(faces, key=lambda f: f.bbox[2] - f.bbox[0])
    x1, y1, x2, y2 = face.bbox
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    h, w = image.shape[:2]
    half = min(max(x2 - x1, y2 - y1) * (1 + margin) / 2, w / 2, h / 2)
    cx = min(max(cx, half), w - half)
    cy = min(max(cy, half), h - half)
    crop = image[int(cy - half):int(cy + half), int(cx - half):int(cx + half)]
    crop = cv2.resize(crop, (CROP, CROP), interpolation=cv2.INTER_CUBIC)
    faces = detector.detect(crop)
    if not faces:
        return None, None
    return crop, face_attrib.predict(crop, max(faces, key=lambda f: f.bbox[2] - f.bbox[0]))


panels = []
for name, label in SUBJECTS:
    crop, result = face_panel(demo_dir / name)
    if crop is None:
        print(f'{name}: no face found')
        continue
    panels.append((cv2.cvtColor(crop, cv2.COLOR_BGR2RGB), result, label))
    print(f'{label:<12} {", ".join(result.labels(THRESHOLD)) or "none"}')

## 2. Visualize Results

**First row**: Original images
**Second row**: Faces annotated with the attributes above the threshold

In [ ]:
fig, axes = plt.subplots(1, len(panels), figsize=(2.9 * len(panels), 4.6))

for ax, (crop, result, label) in zip(axes, panels):
    ax.imshow(crop)
    ax.set_title(label, fontsize=12)
    ax.axis('off')

    # the five heads are independent, so print every probability rather than a winner
    for i, (name, prob) in enumerate(result.as_dict().items()):
        on = prob > THRESHOLD
        ax.text(0.0, -0.05 - i * 0.075, name, transform=ax.transAxes, fontsize=9,
                family='monospace', color='#111' if on else '#999', va='top')
        ax.text(1.0, -0.05 - i * 0.075, f'{"True " if on else "False"} {prob:.2f}',
                transform=ax.transAxes, fontsize=9, family='monospace',
                color='#0a7d3f' if on else '#999', va='top', ha='right')

plt.tight_layout()
plt.show()

## 3. Inspect a Single Prediction

`predict()` returns a `FaceStateResult` and also writes the probabilities back onto the `Face` object (`face.left_eye_open`, `face.right_eye_open`, `face.eyeglasses`, `face.mask`, `face.sunglasses`).

In [ ]:
image = cv2.imread(str(demo_dir / 'state_b_sunglasses.jpg'))
face = detector.detect(image)[0]

result = face_attrib.predict(image, face)

print(result)
print()
for name, prob in result.as_dict().items():
    marker = '✔' if prob > THRESHOLD else '✘'
    print(f"  {marker} {name} {prob:.4f}")

print()
print(f"Face enriched: sunglasses={face.sunglasses:.4f}, left_eye_open={face.left_eye_open:.4f}")

## Notes

- The five values come from independent binary heads. They do not sum to 1, and more than one
  can be high at once, so threshold each separately and never use `argmax`.
- A face in sunglasses usually reads `sunglasses` true and both eyes false, because the lenses
  hide the eyes.